In [1]:
print("hello world")

hello world


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

In [3]:
df = pd.read_csv("train.csv")

In [4]:
df.head()

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56


In [5]:
df.shape

(517754, 14)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517754 entries, 0 to 517753
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      517754 non-null  int64  
 1   road_type               517754 non-null  object 
 2   num_lanes               517754 non-null  int64  
 3   curvature               517754 non-null  float64
 4   speed_limit             517754 non-null  int64  
 5   lighting                517754 non-null  object 
 6   weather                 517754 non-null  object 
 7   road_signs_present      517754 non-null  bool   
 8   public_road             517754 non-null  bool   
 9   time_of_day             517754 non-null  object 
 10  holiday                 517754 non-null  bool   
 11  school_season           517754 non-null  bool   
 12  num_reported_accidents  517754 non-null  int64  
 13  accident_risk           517754 non-null  float64
dtypes: bool(4), float64(

In [7]:
df["road_signs_present"]=df["road_signs_present"].astype(int)
df["public_road"]=df["public_road"].astype(int)
df["holiday"]=df["holiday"].astype(int)
df["school_season"]=df["school_season"].astype(int)


In [8]:
x=df.drop(columns=["accident_risk","id"])
y=df["accident_risk"]



In [9]:
x_train,x_test, y_train, y_test=train_test_split(x,y,test_size=0.3,random_state=42)

In [10]:
x_train.head()

,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents
406072,rural,4,0.67,25,dim,rainy,0,1,afternoon,1,1,3
83732,urban,2,0.73,60,dim,foggy,1,1,evening,0,0,0
262931,rural,4,0.09,45,night,rainy,0,0,evening,0,0,2
634,highway,1,0.71,25,dim,clear,1,0,evening,0,0,1
291964,rural,3,0.84,35,night,foggy,0,0,morning,1,0,0


In [63]:
ohe=OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
ohe_train=ohe.fit_transform(x[["road_type","lighting", "weather", "time_of_day" ]])
ohe_test=ohe.transform(x_test[["road_type","lighting", "weather", "time_of_day" ]])

In [64]:
ohe_train

array([[0., 1., 0., ..., 1., 0., 0.],
       [0., 1., 0., ..., 0., 1., 0.],
       [1., 0., 1., ..., 0., 0., 1.],
       ...,
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 1., 0., 0.]], shape=(517754, 8))

In [65]:
# Combine with numeric columns
X_final_train = np.hstack([
    x[["num_lanes", "curvature", "speed_limit", 
             "road_signs_present", "public_road", 
             "holiday", "school_season", 
             "num_reported_accidents"]].values,
    ohe_train
])

X_final_test = np.hstack([
    x_test[["num_lanes", "curvature", "speed_limit", 
            "road_signs_present", "public_road", 
            "holiday", "school_season", 
            "num_reported_accidents"]].values,
    ohe_test
])

In [66]:
X_final_train

array([[2.0e+00, 6.0e-02, 3.5e+01, ..., 1.0e+00, 0.0e+00, 0.0e+00],
       [4.0e+00, 9.9e-01, 3.5e+01, ..., 0.0e+00, 1.0e+00, 0.0e+00],
       [4.0e+00, 6.3e-01, 7.0e+01, ..., 0.0e+00, 0.0e+00, 1.0e+00],
       ...,
       [4.0e+00, 6.2e-01, 2.5e+01, ..., 0.0e+00, 0.0e+00, 0.0e+00],
       [3.0e+00, 6.3e-01, 2.5e+01, ..., 0.0e+00, 0.0e+00, 0.0e+00],
       [2.0e+00, 3.1e-01, 4.5e+01, ..., 1.0e+00, 0.0e+00, 0.0e+00]],
      shape=(517754, 16))

In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
lr=LinearRegression()
lr.fit(X_final_train,y_train)
lr_pred=lr.predict(X_final_test)
r2_score(y_test,lr_pred)

0.803416519803396

In [16]:
from sklearn.metrics import root_mean_squared_error
root_mean_squared_error(y_test,lr_pred)

0.07358665502141039

In [57]:
from sklearn.ensemble import RandomForestRegressor
rf=RandomForestRegressor(n_estimators=150)
rf.fit(X_final_train,y_train)
rf_pred=rf.predict(X_final_test)
root_mean_squared_error(y_test,rf_pred)

0.059561927079156134

In [25]:
root_mean_squared_error(y_test,rf_pred)

0.0594860986955083

0         0.13
1         0.35
2         0.30
3         0.21
4         0.56
          ... 
517749    0.32
517750    0.26
517751    0.19
517752    0.51
517753    0.22
Name: accident_risk, Length: 517754, dtype: float64

In [68]:
from xgboost import XGBRegressor
xgb= XGBRegressor(
    n_estimators=500,    # ✅ correct
    learning_rate=0.05,
    max_depth=6,
    objective="reg:squarederror")
xgb.fit(X_final_train,y)
xgb_pred=xgb.predict(X_final_test)
root_mean_squared_error(y_test,xgb_pred)

0.05577584641403474

In [34]:
root_mean_squared_error(y_test,xgb_pred)

0.056414613455165796

In [76]:
from sklearn.neighbors import KNeighborsRegressor
knn=KNeighborsRegressor()
knn.fit(X_final_train,y)
knn_pred=knn.predict(X_final_test)
root_mean_squared_error(y_test,knn_pred)

0.06784282640880154

In [72]:
from sklearn.ensemble import AdaBoostRegressor
ada=AdaBoostRegressor(n_estimators=200)
ada.fit(X_final_train,y)
ada_pred=knn.predict(X_final_test)
root_mean_squared_error(y_test,ada_pred)

0.08869107672253872

In [43]:
test_data=pd.read_csv("test.csv")

In [50]:
final_id=test_data["id"]

In [28]:
test_data

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents
0,517754,highway,2,0.34,45,night,clear,True,True,afternoon,True,True,1
1,517755,urban,3,0.04,45,dim,foggy,True,False,afternoon,True,False,0
2,517756,urban,2,0.59,35,dim,clear,True,False,afternoon,True,True,1
3,517757,rural,4,0.95,35,daylight,rainy,False,False,afternoon,False,False,2
4,517758,highway,2,0.86,35,daylight,clear,True,False,evening,False,True,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
172580,690334,rural,2,0.01,45,dim,rainy,False,False,afternoon,True,True,2
172581,690335,rural,1,0.74,70,daylight,foggy,False,True,afternoon,False,False,2
172582,690336,urban,2,0.14,70,dim,clear,False,False,evening,True,True,1
172583,690337,urban,1,0.09,45,daylight,foggy,True,True,morning,False,True,0


In [29]:
test_data["road_signs_present"]=test_data["road_signs_present"].astype(int)
test_data["public_road"]=test_data["public_road"].astype(int)
test_data["holiday"]=test_data["holiday"].astype(int)
test_data["school_season"]=test_data["school_season"].astype(int)

In [30]:
test_data.drop(columns=["id"],inplace=True)

In [31]:
ohe_test_data=ohe.transform(test_data[["road_type","lighting", "weather", "time_of_day" ]])

In [32]:
X_final_test_result = np.hstack([
    test_data[["num_lanes", "curvature", "speed_limit", 
             "road_signs_present", "public_road", 
             "holiday", "school_season", 
             "num_reported_accidents"]].values,
    ohe_test_data
])

In [73]:
ans=ada.predict(X_final_test_result)

array([0.29105696, 0.12110504, 0.18783669, ..., 0.25207126, 0.12902051,
       0.48371693], shape=(172585,), dtype=float32)

In [74]:
submition=pd.DataFrame({
    "id":final_id,
    "accident_risk":ans
})

In [75]:
submition.to_csv("submition.csv",index=False)